# Experiment 3: Joint Adam \((\theta, m)\)

Zhong–Wang Simulation I, Case 1, \(\tau=0.5\).

**Primary estimator:** final joint-Adam \(\hat\theta\) (train \(\theta\) and NN \(m\) together for exactly `EPOCHS` epochs — **no early stopping**).

**Secondary diagnostic only:** freeze \(\hat m\), then convex `linprog` for \(\theta_{\mathrm{profiled}}\). This does **not** replace the primary estimator.

Oracle \(f_0\) and \(\varphi^*\) throughout. Sandwich / \(C_n\) / IF use residualized \(\tilde X\).

**How to use:** Run All below. The next cell loads `run_experiment3.py` (helper). You do not need to run that file separately.


## Why \(m^*=m_0\)

On \(z_j\in[0,2]\), \(\mathrm{ReLU}(z_j)=z_j\). Hence \(m_0(z)=0.56\sum_{j=1}^8 z_j\) lies in the dense ReLU class (widths `[32,32,16]`, no sparsity mask). We take \(m^*=m_0\); observed \(\hat m-m_0\) is finite-sample estimation / optimization error.


## vs Experiment 2

Experiment 2 fits \(m\) with \(\theta\equiv\theta_0\). Experiment 3 adds joint feedback: \(\theta\) errors during training can alter \(m\), and \(m\) errors alter \(\theta\) updates. Comparing \(C_n\) and IF gaps between 2 and 3 diagnoses that feedback.


In [ ]:
import importlib.util
from pathlib import Path

_path = Path("run_experiment3.py").resolve()
_spec = importlib.util.spec_from_file_location("exp3", _path)
exp3 = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(exp3)
globals().update({k: getattr(exp3, k) for k in dir(exp3) if not k.startswith("_")})
print("F0 =", F0)
print("HIDDEN =", HIDDEN, "EPOCHS =", EPOCHS, "(no early stopping)")
print("PRIMARY = joint Adam theta; SECONDARY = profiled linprog after freeze m")
print("cwd =", ROOT)


## Write config (does not run Monte Carlo)


In [ ]:
config = write_config()
print("Wrote", CONFIG_JSON)
print(json.dumps(config, indent=2))


## Monte Carlo loop (run manually)

Restartable. Primary = Adam; profiled columns are secondary diagnostics.


In [ ]:
completed = load_completed()
print(f"Already completed: {len(completed)} rows")

for n in N_VALUES:
    for rep in range(1, Q + 1):
        key = (n, rep)
        if key in completed:
            continue
        row = run_one(n, rep)
        append_result(row)
        completed.add(key)
        if rep % 10 == 0 or rep == 1:
            print(
                f"n={n} rep={rep}/{Q}  "
                f"adam={row['theta1_hat_joint_adam']:.4f},{row['theta2_hat_joint_adam']:.4f}  "
                f"prof={row['theta1_hat_profiled']:.4f},{row['theta2_hat_profiled']:.4f}  "
                f"nuisance_L2={row['nuisance_L2']:.4f}  Cn_norm={row['Cn_norm']:.3f}"
            )

print("Done. Results at", RESULTS_CSV)


## Summary (primary Adam + separate profiled block)


In [ ]:
if not RESULTS_CSV.exists():
    raise FileNotFoundError("Run the Monte Carlo cell first")

df = pd.read_csv(RESULTS_CSV)
summary = summarize(df)
summary.to_csv(SUMMARY_CSV, index=False)
print(summary.to_string(index=False))
print("Wrote", SUMMARY_CSV)


## Figures


In [ ]:
df = pd.read_csv(RESULTS_CSV)
make_figures(df)
print("Figures written to", FIG_DIR)
for p in sorted(FIG_DIR.glob("*.png")):
    print(" ", p.name)
